In [1]:
def ResponseMatrix(self,sample,acceptance,fullsel,vart,varr,bin_edges,wlab,potw,univ=-1,wvar=''):
        #get number of signal events at true level before selection
        truevals = sample.query(acceptance,engine="python")[vart]
        tweights = sample.query(acceptance,engine="python")[wlab]*potw
        if univ>=0:
            vweights = sample.query(acceptance,engine="python")[wvar]
            if (np.stack(vweights).ndim>1):
                #multisim, pick specific universe
                vweights = np.stack(vweights)[:,univ]/1000.
            vweights[np.isnan(vweights)] = 1
            vweights[vweights > 100] = 1
            vweights[vweights < 0] = 1
            vweights[vweights == np.inf] = 1
            tweights = tweights*vweights
        n, bins = np.histogram(truevals,weights=tweights,bins=bin_edges)
        #get number of signal events at reco and true level after selection
        x = sample.query(acceptance+' and '+fullsel,engine="python")[vart]
        y = sample.query(acceptance+' and '+fullsel,engine="python")[varr]
        w = sample.query(acceptance+' and '+fullsel,engine="python")[wlab]*potw
        if univ>=0:
            vw = sample.query(acceptance+' and '+fullsel,engine="python")[wvar]
            if (np.stack(vw).ndim>1):
                #multisim, pick specific universe
                vw = np.stack(vw)[:,univ]/1000.
            vw[np.isnan(vw)] = 1
            vw[vw > 100] = 1
            vw[vw < 0] = 1
            vw[vw == np.inf] = 1
            w = w*vw
        H, xb, yb = np.histogram2d(x,y,weights=w,bins=[bin_edges,bin_edges])
        #get response matrix
        rm = np.transpose(H)/n
        return rm, xb, yb

In [ ]:
def sys_err_unisim_with_resp_func(self, signame, var_name, true_var_name, query, acceptance, x_range, n_bins):
        #this is not done for lee

        if x_range == None:
            bins = n_bins
        else:
            bins = np.linspace(x_range[0],x_range[1],n_bins+1)

        n_bins = len(bins)-1

        weightVarCV = "weightSplineTimesTune"
        weightVarVar = "weightSpline"

        # assume list of knobs is fixed. If not we will get errors
        # knobRPA [up,dn]
        # knobCCMEC [up,dn]
        # knobAxFFCCQE [up,dn]
        # knobVecFFCCQE [up,dn]
        # knobDecayAngMEC [up,dn]
        # knobThetaDelta2Npi [up,dn]
        knob_v = ['knobRPA','knobCCMEC','knobAxFFCCQE','knobVecFFCCQE','knobDecayAngMEC','knobThetaDelta2Npi']
        knob_n = [2,1,1,1,1,1]

        n_tot_v = []
        for u,knob in enumerate(knob_v):
            n_tot_v.append( np.empty([ knob_n[u] ,n_bins]) )
            n_tot_v[-1].fill(0)

        n_cv_tot = np.empty(n_bins)
        n_cv_tot.fill(0)

        for t in self.samples:
            if t in ["ext", "data", "data_7e18", "data_1e20","lee"]:
                continue

            tree = self.samples[t]

            extra_query = ""
            if t == "mc":
                extra_query = "& " + self.nu_pdg
            if t == signame:
                extra_query = "& ~(" + acceptance +")"

            queried_tree = tree.query(query+extra_query,engine="python")
            variable = queried_tree[var_name]
            spline_fix_cv  = queried_tree[weightVarCV] * self.weights[t]
            spline_fix_var = queried_tree[weightVarVar] * self.weights[t]

            n_cv, bins = np.histogram(variable,bins=bins,weights=spline_fix_cv)
            n_cv_tot += n_cv

            for n,knob in enumerate(knob_v):

                weight_up = queried_tree['%sup'%knob].values
                weight_up[np.isnan(weight_up)] = 1
                weight_up[weight_up > 100] = 1
                weight_up[weight_up < 0] = 1
                weight_up[weight_up == np.inf] = 1
                n_up, bins = np.histogram(variable, weights=weight_up * spline_fix_var,bins=bins)
                n_tot_v[n][0] += n_up

                if (knob_n[n] == 2):
                    weight_dn = queried_tree['%sdn'%knob].values
                    weight_dn[np.isnan(weight_dn)] = 1
                    weight_dn[weight_dn > 100] = 1
                    weight_dn[weight_dn < 0] = 1
                    weight_dn[weight_dn == np.inf] = 1
                    n_dn, bins = np.histogram(variable, weights=weight_dn * spline_fix_var,bins=bins)
                    n_tot_v[n][1] += n_dn

        # do stuff for signal sample
        tree = self.samples[signame]

        extra_query = ""
        if signame == "mc":
            extra_query += "& " + self.nu_pdg

        queried_tree = tree.query(query+"& (" + acceptance +")"+extra_query,engine="python")
        variable = queried_tree[var_name]
        #print ('N universes is :',len(syst_weights))
        spline_fix_cv  = queried_tree[weightVarCV] * self.weights[signame]

        n_cv, bins = np.histogram(variable,bins=bins,weights=spline_fix_cv)
        n_cv_tot += n_cv

        true_variable = tree.query(acceptance,engine="python")[true_var_name]
        spline_fix_cv  = tree.query(acceptance,engine="python")[weightVarCV] * self.weights[signame]
        t_cv, bins = np.histogram(true_variable,bins=bins,weights=spline_fix_cv)

        for n,knob in enumerate(knob_v):

                rmv_up, xb, yb = self.ResponseMatrix(tree,acceptance,query,true_var_name,var_name,\
                                                     bins,weightVarVar,self.weights[signame],0,'%sup'%knob)
                rp_up = rmv_up.dot(t_cv)
                n_tot_v[n][0] += rp_up
                if (knob_n[n] == 2):
                    rmv_dn, xb, yb = self.ResponseMatrix(tree,acceptance,query,true_var_name,var_name,\
                                                         bins,weightVarVar,self.weights[signame],0,'%sdn'%knob)
                    rp_dn = rmv_dn.dot(t_cv)
                    n_tot_v[n][1] += rp_dn

        cov = np.empty([len(n_cv), len(n_cv)])
        cov.fill(0)

        for n,knob in enumerate(knob_v):

            this_cov = np.empty([len(n_cv), len(n_cv)])
            this_cov.fill(0)

            #print ('knob : %s has :'%knob)
            #print ('n_cv : ',n_cv_tot)
            #print ('n_up : ',n_tot_v[n][0])
            #print ('n_dn : ',n_tot_v[n][1])

            if (knob_n[n] == 2):
                for i in range(len(n_cv)):
                    #print ('knob %s has CV: %.0f, VAR UP: %.0f, VAR DN: %.0f entries'%(knob,n_cv_tot[i],n_tot_v[n][0][i],n_tot_v[n][1][i]))
                    for j in range(len(n_cv)):
                        this_cov[i][j] += (n_tot_v[n][0][i] - n_cv_tot[i]) * (n_tot_v[n][0][j] - n_cv_tot[j])
                        this_cov[i][j] += (n_tot_v[n][1][i] - n_cv_tot[i]) * (n_tot_v[n][1][j] - n_cv_tot[j])
                this_cov /= 2.

            if (knob_n[n] == 1):
                for i in range(len(n_cv)):
                    #print ('knob %s has CV: %.0f, VAR: %.0f entries'%(knob,n_cv_tot[i],n_tot_v[n][0][i]))
                    for j in range(len(n_cv)):
                        this_cov[i][j] += (n_tot_v[n][0][i] - n_cv_tot[i]) * (n_tot_v[n][0][j] - n_cv_tot[j])

            cov += this_cov

        return cov

In [2]:
def sys_err_with_resp_func(self, wname, signame, var_name, true_var_name, query, acceptance, x_range, n_bins, weightVar, maxUniv = False):
        #this is not done for lee

        if x_range == None:
            bins = n_bins
        else:
            bins = np.linspace(x_range[0],x_range[1],n_bins+1)

        weightVarCV = weightVar
        #need special case since genie tune changed at some point
        weightVarVar = weightVar
        if (wname == "weightsGenie"): weightVarVar = "weightSpline"

        # how many universes?
        Nuniverse = 100 #len(df)
        if maxUniv:
            if (wname == "weightsGenie"): Nuniverse = 500
            if (wname == "weightsFlux"):  Nuniverse = 1000
            if (wname == "weightsReint"): Nuniverse = 1000
        n_bins = len(bins)-1

        n_tot = np.empty([Nuniverse, n_bins])
        n_cv_tot = np.empty(n_bins)
        n_tot.fill(0)
        n_cv_tot.fill(0)

        for t in self.samples:
            if t in ["ext", "data", "data_7e18", "data_1e20","lee"]:
                continue

            tree = self.samples[t]

            extra_query = ""
            if t == "mc":
                extra_query = "& " + self.nu_pdg # "& ~(abs(nu_pdg) == 12 & ccnc == 0) & ~(npi0 == 1 & category != 5)"
            if t == signame:
                extra_query = "& ~(" + acceptance +")"

            queried_tree = tree.query(query+extra_query,engine="python")
            variable = queried_tree[var_name]
            syst_weights = queried_tree[wname]
            #print ('N universes is :',len(syst_weights))
            spline_fix_cv  = queried_tree[weightVarCV] * self.weights[t]
            spline_fix_var = queried_tree[weightVarVar] * self.weights[t]

            s = syst_weights

            df = pd.DataFrame(s.values.tolist())
            #print (df)
            #print(t,wname,np.shape(df))
            #continue

            n_cv, bins = np.histogram(
                variable,
                bins=bins,
                weights=spline_fix_cv)
            n_cv_tot += n_cv

            if not df.empty:
                for i in range(Nuniverse):
                    weight = df[i].values / 1000.
                    weight[np.isnan(weight)] = 1
                    weight[weight > 100] = 1
                    weight[weight < 0] = 1
                    weight[weight == np.inf] = 1

                    n, bins = np.histogram(
                        variable, weights=weight*spline_fix_var,bins=bins)
                    n_tot[i] += n

        # do stuff for signal sample
        tree = self.samples[signame]

        extra_query = ""
        if signame == "mc":
            extra_query += "& " + self.nu_pdg

        queried_tree = tree.query(query+"& (" + acceptance +")"+extra_query,engine="python")
        variable = queried_tree[var_name]
        syst_weights = queried_tree[wname]
        #print ('N universes is :',len(syst_weights))
        spline_fix_cv  = queried_tree[weightVarCV] * self.weights[signame]

        s = syst_weights

        df = pd.DataFrame(s.values.tolist())
        #print (df)
        #print(t,wname,np.shape(df))
        #continue

        #print(bins)
        #print(variable)
        #print(spline_fix_cv)
        n_cv, bins = np.histogram(
            variable,
            bins=bins,
            weights=spline_fix_cv)
        n_cv_tot += n_cv
        #print('n_cv',n_cv)

        true_variable = tree.query(acceptance,engine="python")[true_var_name]
        spline_fix_cv  = tree.query(acceptance,engine="python")[weightVarCV] * self.weights[signame]
        t_cv, bins = np.histogram(
            true_variable,
            bins=bins,
            weights=spline_fix_cv)
        #print('t_cv',t_cv)

        if not df.empty:
            for i in range(Nuniverse):
                rmv, xb, yb = self.ResponseMatrix(tree,acceptance,query,true_var_name,var_name,\
                                                  bins,weightVarVar,self.weights[signame],i,wname)
                #print(rmv)
                rp = rmv.dot(t_cv)
                #print("variation: ",rp)
                n_tot[i] += rp

        # now compute the covariance
        cov = np.empty([len(n_cv), len(n_cv)])
        cov.fill(0)

        for n in n_tot:
            for i in range(len(n_cv)):
                for j in range(len(n_cv)):
                    cov[i][j] += (n[i] - n_cv_tot[i]) * (n[j] - n_cv_tot[j])

        cov /= Nuniverse

        return cov

In [1]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append("../../../")
import data_loading as dl
from importlib import reload
reload(dl)

from microfit import run_plotter as rp
from microfit import histogram as hist

from microfit import variable_definitions as vdef
from microfit import selections

In [2]:
keep_vars = [
    "Signal_1e1p", "mc_signal_1e1p", "nu_pdg", "TrueElecIdx", "TrueLeadProtonIdx", "InFV", "HasNoMesons",
    "TrueNElec", "TrueNProt", "TrueDeltaPT", "TrueDeltaAlphaT", "TruePN", "TrueAlpha3D",
    "nproton", "npion", "npi0", "nelec", "nmuon", "isVtxInFiducial",
    "Sel_1e1p", "sel_1e1p_w_cuts", "RecoElectronCandidateIdx", "RecoLeadProtonCandidateIdx", "InFV_reco",
    "RecoElecPassMomCut", "RecoLeadProtonPassMomCut", "n_reco_tracks", "n_reco_showers",
    "RecoDeltaPT", "RecoDeltaAlphaT", "RecoPN", "RecoAlpha3D", "RecoECal", "Reco_mag_q", "RecoPL",
    "nslice", "selected", "shr_energy_tot_cali", "_opfilter_pe_beam", "_opfilter_pe_veto", "bnbdata", "extdata",
    "CosmicIPAll3D", "hits_ratio", "shrmoliereavg", "subcluster", "trkfit", "trkshrhitdist2", "tksh_distance",
    "shr_tkfit_nhits_tot", "shr_tkfit_dedx_max", "tksh_angle", "shr_trk_len", "reco_e",
    "RecoLeadProton_trk_len", "RecoLeadProton_trk_trunk_dEdx_y", "RecoLeadProton_dEdx_y_per_trklen",
    "RecoLeadProtonCandidate_trk_pid", "RecoElectronCandidate_shr_pid", "RecoElectron_conversion_dist",
    "pi0_radlen1", "pi0_radlen2", "pi0_score", "nonpi0_score", "bkg_score",
    "RecoElecE", "RecoElecModMom", "RecoElecMomX", "RecoElecMomY", "RecoElecMomZ",
    "RecoLeadProtonKE", "RecoLeadProtonModMom", "RecoLeadProtonMomX", "RecoLeadProtonMomY", "RecoLeadProtonMomZ",
    "ccnc",
]

In [3]:
RUN = ["3"]
#RUN = ["1","2","3","4a","4b","4c","4d","5","1A_OT","1B_OT"] # for detvars with bnb or for closure test
#RUN = ["1","2","3","4c","5"] # for nuwro_fd, no run 4b and 4d available
blinded = False
#data="nuwro_fd"
data="bnb"

In [4]:
rundata, mc_weights, data_pot = dl.load_runs(
    RUN,
    data=data,
    loadpi0variables=False,
    loadshowervariables=True,
    loadrecoveryvars=False,
    loadsystematics=True,
    numupresel=False,
    loadnumuvariables=False,
    use_bdt=True,
    load_lee=False,
    load_nue_tki=True,
    keep_columns=keep_vars,
    blinded=blinded,
    load_crt_vars=False,
    enable_cache=True,
)

Loading run 3
Updating keep_columns with truth-filtering variables: {'nu_pdg', 'ccnc'}
/exp/uboone/data/users/mmoudgal/PELEE_2023_Samples/run3/run3_bnb_beam_on_crtremerging_pandora_reco2_run3_ana.root


: 

In [9]:
run_combo = "Run"
for run in RUN:
    run_combo += run

In [33]:
print(mc_weights)
print(rundata.keys())

{'data': 1.0, 'ext': 0.2962600445651671, 'mc': 0.19603053435114504, 'nue': 0.0033050193050193047, 'drt': 0.7853211009174312}
dict_keys(['data', 'ext', 'mc', 'nue', 'drt'])


In [40]:
rundata["drt"].loc[rundata["drt"]["category_1e1p"] == 12]

,pi0truth_gamma1_edep,pion_e,true_nu_vtx_sce_z,shr_tkfit_npointsvalid,truthFiducial,shr_px,knobCCMECup,knobDecayAngMECup,shr_energy_tot_cali,shr_tkfit_gap10_dedx_Y,...,dz,dr,paper_category,paper_category_xsec,paper_category_numu,category_1e1p,mc_signal_1e1p,dataset,weights,weights_no_tune
entry,,,,,,,,,,,,,,,,,,,,,


In [32]:
print("weightsGenie" in rundata["mc"].columns)
print(len(rundata["mc"]["weightsGenie"][0]))
rundata["nue"].loc[:, ("weightsGenie", "weightSpline", "weights", "weightSplineTimesTune", "weights_no_tune")]

True
100


,weightsGenie,weightSpline,weights,weightSplineTimesTune,weights_no_tune
entry,,,,,
0,"[1914, 405, 1060, 730, 505, 1791, 1066, 829, 8...",0.981016,0.003779,1.143481,0.003242
1,"[1295, 865, 746, 1131, 1996, 1225, 1211, 1091,...",1.038154,0.004095,1.239054,0.003431
2,"[2081, 892, 641, 1719, 1549, 1732, 1722, 1235,...",0.974409,0.004102,1.241263,0.003220
3,"[1195, 927, 769, 1116, 1683, 1192, 1162, 1152,...",1.648609,0.006388,1.932914,0.005449
4,"[1209, 986, 863, 1131, 1634, 1220, 1167, 1169,...",0.965846,0.003852,1.165619,0.003192
...,...,...,...,...,...
61952,"[1269, 1180, 915, 1295, 730, 1442, 1061, 1255,...",0.964037,0.003737,1.130834,0.003186
61953,"[980, 424, 542, 1388, 948, 550, 1293, 494, 142...",0.965515,0.003643,1.102224,0.003191
61954,"[1499, 348, 1308, 543, 481, 1269, 901, 596, 62...",0.974858,0.003107,0.939972,0.003222


In [5]:
s = rundata["mc"]["weightsGenie"]
df = pd.DataFrame(s.values.tolist())
print(df[1].values)
df.head(20)

NameError: name 'rundata' is not defined

In [ ]:
s = rundata["mc"]["weightsGenie"]
s.head()

In [ ]:
# print(np.stack(s).ndim>1)
if (np.stack(s).ndim>1):
    #multisim, pick specific universe
    p = np.stack(s)[:,0]/1 #000.
    print(p)
print(p)

False


NameError: name 'p' is not defined

In [41]:
rundata["mc"].loc[:, ("knobRPAup",
        "knobRPAdn",
        "knobCCMECup",
        "knobCCMECdn",
        "knobAxFFCCQEup",
        "knobAxFFCCQEdn",
        "knobVecFFCCQEup",
        "knobVecFFCCQEdn",
        "knobDecayAngMECup",
        "knobDecayAngMECdn",
        "knobThetaDelta2Npiup",
        "knobThetaDelta2Npidn",)]

,knobRPAup,knobRPAdn,knobCCMECup,knobCCMECdn,knobAxFFCCQEup,knobAxFFCCQEdn,knobVecFFCCQEup,knobVecFFCCQEdn,knobDecayAngMECup,knobDecayAngMECdn,knobThetaDelta2Npiup,knobThetaDelta2Npidn
entry,,,,,,,,,,,,
0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0
1,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0
2,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0
3,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
32181,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.397231,1.0
32182,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.650770,1.0
32183,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.000000,1.0


In [46]:
s = rundata["mc"]["knobThetaDelta2Npiup"]
#s.head()
df = pd.DataFrame(s.values.tolist())
print(df[0].values)
df.head()

[1.         1.         1.         ... 1.         2.60126025 3.8264348 ]


,0
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0


In [48]:
rundata["mc"].loc[:, "knobThetaDelta2Npiup"].head()

entry
0    1.0
1    1.0
2    1.0
3    1.0
4    1.0
Name: knobThetaDelta2Npiup, dtype: float64